# Notebook initialization

In [ ]:
import pyxdf
import numpy as np
import os
import logging
from PIL import Image


# # for the tests
# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib.widgets import SpanSelector

# from scipy.signal import butter, filtfilt, find_peaks


# and set to true for testing (default) but can be already set to false from outside (e.g., by the test script)
if "doRunTests" not in globals():
    doRunTests = True

if doRunTests:
    import matplotlib.pyplot as plt
    # the best for debug-test plots (external window that you can make fullscreen and zoom)
    %matplotlib qt


    # functions for plotting the tests... 
    def plot_before_after(time_before, data_before, time_after, data_after, title_txt=""):
        """
        Plot the data before and after doing some changes.
        """
        plt.figure()
        plt.plot(time_before, data_before, "+-", label="before")
        plt.plot(time_after, data_after, "*", label="after")
        plt.title(f"Data: before and after {title_txt}")
        plt.xlabel("Time (s)")
        plt.ylabel("Position (m)")
        plt.legend()
        plt.show()

# Classes to manage XDF files

In [ ]:
class XDF_file:
    """
    Class to handle XDF files.
    """

    def __init__(
        self,
        xdf_fullFname: str | os.PathLike,
        select_streams=None,
    ):
        """
        Initialize the XDF file.
        Parameters
        ----------
        xdf_fullFname : str
            Full path to the XDF file.
        select_streams : list of str (as in load_xdf)
            List of stream types to select. If None, all streams are selected.
        """
        self.xdf_fullFname = xdf_fullFname
        self.xdf_fname = os.path.basename(xdf_fullFname)
        self.xdf_dir = os.path.dirname(xdf_fullFname)
        xdf_streams, xdf_header = pyxdf.load_xdf(
            xdf_fullFname,
            select_streams=select_streams,
            synchronize_clocks=True,
            dejitter_timestamps=False,  # to get the raw timestamps to compare with the CSV
            verbose=False,
        )

        self.streams = []
        for i in range(len(xdf_streams)):
            stream = xdf_streams[i]
            self.streams.append(XDF_stream(stream))

    def __str__(self):
        """Print the names and types of all streams in the xdf file"""
        s = ""
        for i in range(len(self.streams)):
            stream = self.streams[i]
            s_type = stream["info"]["type"][0]
            s_name = stream["info"]["name"][0]
            s += f"Stream {i}: {s_type}, {s_name}\n"
        return s

    def get_stream_index(self, searched_stream_type, searched_stream_names):
        """Get the index of the stream of type 'searched_stream_type' AND with name in
        'searched_stream_names'"""

        if not isinstance(searched_stream_names, list):
            # if we get a string (only one name)
            searched_stream_names = [searched_stream_names]

        found_streams = []
        for i, stream in enumerate(self.streams):
            if searched_stream_type == stream.type:
                for searched_stream_name in searched_stream_names:
                    if searched_stream_name == stream.name:
                        found_streams.append(i)

        if not found_streams:
            return None

        if len(found_streams) > 1:
            found_streams_names = [self.streams[i].name for i in found_streams]
            msg = (
                f"Found multiple streams: "
                f"[{searched_stream_type},{found_streams_names}]."
            )
            raise ValueError(msg)

        return found_streams[0]


class XDF_channel:
    """
    Class to handle XDF channels.
    """

    def __init__(self, index, stream):
        self.index = index
        desc = stream["info"]["desc"][0]["channels"][0]["channel"][index]
        self.label = desc["label"][0]
        self.type = desc["type"][0]
        self.unit = desc["unit"][0]

        self.time_series = stream["time_series"][:, index]
        self.time_stamps = stream["time_stamps"]

    def __str__(self):
        """Print the name and type of the channel"""
        s = f"Channel {self.index}: {self.label} ({self.type},  {self.unit})\n"
        return s


class XDF_stream:
    """
    Class to handle XDF streams.
    A stream can be organized by channel (stream.channels = [...],  e.g. for data) or
    not organized (stream.channels = [], e.g. for markers ).
    """

    def __init__(self, xdf_stream):
        self.xdf_stream = xdf_stream
        self.time_stamps = xdf_stream["time_stamps"]
        self.time_series = xdf_stream["time_series"]
        self.name = xdf_stream["info"]["name"][0]
        self.type = xdf_stream["info"]["type"][0]
        self.channels = self.set_channels()

    def __str__(self):
        """Print the names and types of all channels in the stream"""
        s = f"Stream {self.name} ({self.type})\n"
        for i in range(len(self.channels)):
            channel_name = self.channels[i].label
            channel_type = self.channels[i].type
            channel_unit = self.channels[i].unit
            s += f"Channel {i}: {channel_name} ({channel_type}, {channel_unit})\n"
        return s

    def set_channels(self):
        """Set the channels from the xdf stream"""
        channels = []
        try:
            n_channels = len(
                self.xdf_stream["info"]["desc"][0]["channels"][0]["channel"]
            )
            for i in range(n_channels):
                channel = XDF_channel(i, self.xdf_stream)
                channels.append(channel)
        except Exception:
            pass

        return channels

    def get_one_channel(self, name):
        for i in range(len(self.channels)):
            channel = self.channels[i]
            if channel.label == name:
                return channel
        return None

    def get_channel_index(self, channel_name):
        """Get the index of one channel from the stream by its name"""
        channel_index = -1
        nb_channels = len(self.xdf_stream["info"]["desc"][0]["channels"][0]["channel"])
        for i in range(nb_channels):
            current_name = self.xdf_stream["info"]["desc"][0]["channels"][0]["channel"][
                i
            ]["label"][0]
            if current_name == channel_name:
                channel_index = i
                break
        if channel_index == -1:
            return None

        return channel_index


def interpolate_to_constant_time_step(t, x, dt=1 / 30):
    """Interpolate the data to a constant time step"""

    n_columns = x.shape[1] if x.ndim > 1 else 1

    t_new = np.arange(t[0], t[-1], dt)

    if x.ndim < 2:
        x_new = np.interp(t_new, t, x)
    else:
        x_new = np.zeros((len(t_new), n_columns))
        for i in range(n_columns):
            x_new[:, i] = np.interp(t_new, t, x[:, i])

    return x_new, t_new

In [ ]:
if doRunTests:
    xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P04/V1/Reaching/C01P04_QueAla_20210531_1_r.xdf"  # no mouse marker csv file associated -> nan time correction - solution found

    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    print(xdf_file.streams[2].get_one_channel("SpineBase_X"))

In [ ]:
def get_time_correction(xdf_file: XDF_file):
    """Get the time correction from the xdf file name"""

    xdf_fullFname = str(xdf_file.xdf_fullFname)
    is_log_present = False
    log_fullFname = os.path.join(
        os.path.dirname(xdf_fullFname), "..", "kinect_time_correction.log"
    )
    log_fullFname = os.path.normpath(log_fullFname)
    if os.path.exists(log_fullFname):
        is_log_present = True

    is_correction_present = False
    time_correction_file = xdf_fullFname.replace(".xdf", "_xdf_time_correction.csv")
    if os.path.exists(time_correction_file):
        is_correction_present = True

    if not is_log_present:
        raise FileNotFoundError(
            f"The file {log_fullFname} does not exist. Please run the script to generate it."
        )

    if not is_correction_present:
        time_correction = np.float64(0)
    else:
        time_correction = np.loadtxt(time_correction_file, delimiter=",", skiprows=1)

    return time_correction


def make_kinect_time_correction(xdf_file: XDF_file):
    """Make the time correction for the Kinect streams in the xdf file."""

    # get the time correction from the xdf file name
    time_correction = get_time_correction(xdf_file)

    if time_correction == 0:
        # no time correction needed
        return

    if np.isnan(time_correction):
        # this can happen when the mouse csv file is not present...
        logging.warning(
            f"Time correction is NaN. Is there a mouse csv file for {xdf_file.xdf_fullFname}?"
        )
        return

    # here, we are OK to make the time correction
    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")
    i_k_mk = xdf_file.get_stream_index("Markers", "EuroMov-Markers-Kinect")

    if i_k_mo is None or i_k_mk is None:
        raise ValueError(
            f"Cannot find the Kinect streams in {xdf_file.xdf_fullFname}. "
            f"Please check the stream names."
        )

    xdf_file.streams[i_k_mo].time_stamps += time_correction
    xdf_file.streams[i_k_mk].time_stamps += time_correction
    logging.info(
        f"Time correction of {time_correction} s applied to the Kinect streams."
    )

# Remove all rows filled with only zeros in the kinect data

In [ ]:
def remove_zero_rows(xdf_file: XDF_file):
    """
    Remove the rows filled with zeros from the kinect data.
    """

    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")

    if i_k_mo:
        kinect_t = xdf_file.streams[i_k_mo].time_stamps
        kinect_data = xdf_file.streams[i_k_mo].time_series

        # find the indexes of kinect data that are filled with zeros
        zero_rows = np.all(kinect_data == 0, axis=1)
        zero_rows_indices = np.where(zero_rows)[0]

        if len(zero_rows_indices) > 0:
            # remove the zero rows from the data
            kinect_data = np.delete(kinect_data, zero_rows_indices, axis=0)
            kinect_t = np.delete(kinect_t, zero_rows_indices, axis=0)

            # modify the stream (but not the original xdf_file.xdf_streams )
            xdf_file.streams[i_k_mo].time_series = kinect_data
            xdf_file.streams[i_k_mo].time_stamps = kinect_t

            logging.info(
                f"Removed {len(zero_rows_indices)} rows filled with only zeros in the Kinect data"
            )


if doRunTests:

    # test the function
    xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P01/V1/Reaching/001_BenMus_20210202_1_r.xdf"  # large block of zeros in the beginning -> incomplete data - no solution

    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")
    if i_k_mo is None:
        raise ValueError("No MoCap stream found")

    i_wz = xdf_file.streams[i_k_mo].get_channel_index("WristRight_Z")

    t_before = xdf_file.streams[i_k_mo].time_stamps.copy()
    data_before = xdf_file.streams[i_k_mo].time_series[:, i_wz].copy()

    remove_zero_rows(xdf_file)

    t_after = xdf_file.streams[i_k_mo].time_stamps
    data_after = xdf_file.streams[i_k_mo].time_series[:, i_wz]

    plot_before_after(t_before, data_before, t_after, data_after)


# Interpolate the Mocap data
This is mandatory because the kinect and mouse data are produced by the computer: the sampling rate is not waranted to be constant (samples are forgetten... sometimes). 

In [ ]:
def resample_kinect_data(xdf_file: XDF_file):
    """
    Resample the Kinect data to a constant time step.
    """

    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")
    if i_k_mo:
        kinect_t = xdf_file.streams[i_k_mo].time_stamps
        kinect_data = xdf_file.streams[i_k_mo].time_series

        inital_sampling_rate = 1 / (kinect_t[1] - kinect_t[0])

        # resample the data to a constant time step
        time_step = 1 / 30
        kinect_data, kinect_t = interpolate_to_constant_time_step(
            kinect_t, kinect_data, dt=time_step
        )

        # modify the stream (but not the original xdf_file.xdf_streams )
        xdf_file.streams[i_k_mo].time_series = kinect_data
        xdf_file.streams[i_k_mo].time_stamps = kinect_t

        logging.info(
            f"Kinect resampled at {1/time_step:3.2f} Hz (from about {inital_sampling_rate:3.2f} Hz before)."
        )


if doRunTests:
    xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P01/V1/Reaching/001_BenMus_20210202_1_r.xdf"  # large block of zeros in the beginning -> incomplete data - no solution
    # NOTE: the sampling rate is 15hz in this xdf file

    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")

    if i_k_mo:
        remove_zero_rows(xdf_file)

        stream = xdf_file.streams[i_k_mo]
        i_wz = stream.get_channel_index("WristRight_Z")
        if i_wz is None:
            raise ValueError("No WristRight_Z channel found")
        t_before = stream.time_stamps.copy()
        data_before = stream.time_series[:, i_wz].copy()

        resample_kinect_data(xdf_file)

        t_after = stream.time_stamps
        data_after = stream.time_series[:, i_wz]
        plot_before_after(t_before, data_before, t_after, data_after)

# Compute panu for one xdf file


In [ ]:
def save_panu(xdf_fullFname):
    fname_xdf = os.path.basename(xdf_fullFname)
    fname_panu = fname_xdf.replace(".xdf", "_xdf_panu.csv")

    # load the xdf file
    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )

    make_kinect_time_correction(xdf_file)
    remove_zero_rows(xdf_file)
    resample_kinect_data(xdf_file)

    logging.info(f"Saved '{fname_panu}'")

# Compute panu for one visit

In [ ]:
def get_xdf_files_in_visit(visit_dir, directories_to_skip=None):
    """Get the xdf files in the visit_dir"""

    xdf_files = []

    if not os.path.exists(visit_dir):
        raise ValueError(f"Directory {visit_dir} does not exist")

    for root, dirs, files in os.walk(visit_dir):
        # Skip the directories that are in the directories_to_skip list
        if directories_to_skip and any(
            skip_dir in root for skip_dir in directories_to_skip
        ):
            continue
        for file in files:
            if directories_to_skip and any(
                skip_dir in root for skip_dir in directories_to_skip
            ):
                continue
            if file.endswith(".xdf"):
                xdf_files.append(os.path.join(root, file))

    if not xdf_files:
        logging.warning(f"No xdf files found in {visit_dir}")

    return xdf_files


def is_already_done_panu_in_visit(visitPath, checkLog_fname):
    """
    Check if the visit was already processed with panu
    """
    absVisitPath = os.path.abspath(visitPath)
    full_checkLog_fname = os.path.join(absVisitPath, checkLog_fname)
    return os.path.isfile(full_checkLog_fname)


def create_panu_log_file(visitPath, checkLog_fname):
    """
    Create the log file for panu
    """
    absVisitPath = os.path.abspath(visitPath)
    full_checkLog_fname = os.path.join(absVisitPath, checkLog_fname)

    # Create the log file
    logging.basicConfig(
        filename=full_checkLog_fname,
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s",
        force=True,  # remove previous handlers and set the new one
    )

    return full_checkLog_fname


def merge_panu_png_files_to_pdf(visit_path):
    """
    Merge the panu png files in the visit folder
    """
    png_files = [f for f in os.listdir(visit_path) if f.endswith("_panu.png")]
    png_files.sort()

    if len(png_files) > 1:
        # read the png files
        images = [
            Image.open(os.path.join(visit_path, png_file)) for png_file in png_files
        ]
        # convert to RGB
        images = [img.convert("RGB") for img in images]

        images[0].save(
            os.path.join(visit_path, f"{os.path.basename(visit_path)}_panu_png.pdf"),
            save_all=True,
            append_images=images[1:],
        )

        # remove the original png files
        for png_file in png_files:
            os.remove(os.path.join(visit_path, png_file))
            # print(f"    Removed {png_file}")
    else:
        msg = "No panu png files to merge"
        logging.info(msg)
        print(msg)


def merge_panu_pdf_files_to_pdf(visit_path):
    """
    Merge the panu pdf files in the visit folder
    """
    pdf_files = [f for f in os.listdir(visit_path) if f.endswith("_panu.pdf")]
    pdf_files.sort()

    from pypdf import PdfWriter

    if len(pdf_files) > 1:
        # read the pdf files
        pdf_merger = PdfWriter()
        for pdf_file in pdf_files:
            pdf_merger.append(os.path.join(visit_path, pdf_file))

        pdf_merger.write(
            os.path.join(visit_path, f"{os.path.basename(visit_path)}_panu_pdf.pdf")
        )
        pdf_merger.close()

        # remove the original pdf files
        for pdf_file in pdf_files:
            os.remove(os.path.join(visit_path, pdf_file))
            # print(f"    Removed {pdf_file}")
            pass
    else:
        msg = "No panu pdf files to merge"
        logging.info(msg)
        print(msg)


def get_panu_in_visit(visit_dir, directories_to_skip=None):
    """Correct the kinect timestamps for all the xdf files in the visit_dir"""

    panu_log = "panu.log"
    xdf_files = get_xdf_files_in_visit(visit_dir, directories_to_skip)

    if not xdf_files or len(xdf_files) == 0:
        return

    if is_already_done_panu_in_visit(visit_dir, panu_log):
        print(f"    Already done: '{panu_log}' found")
        return

    create_panu_log_file(visit_dir, panu_log)
    logging.info(f"Starting panu in {visit_dir}")

    for xdf_fullFname in xdf_files:
        print(f"---- \n{xdf_fullFname}")
        logging.info(f"{os.path.basename(xdf_fullFname)}")
        save_panu(xdf_fullFname)

    merge_panu_png_files_to_pdf(os.path.dirname(visit_dir))
    logging.info("panu completed")
    print(f"    panu completed: see '{panu_log}' for details")


if doRunTests:
    visit_dir = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1"
    visit_dir = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210419_V2"
    visit_dir = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210715_V3"
    visit_dir = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20210716_V1"
    # visit_dir = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20210820_V2"
    # visit_dir = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20211116_V3"

    visit_dir = "../dat/ReArm.lnk/C1P42/V1"
    visit_dir = "../dat/ReArm.lnk/C1P42/V2"
    visit_dir = "../dat/ReArm.lnk/C1P42/V3"

    visit_dir = (
        "../dat/ReArm.lnk/DATA_named/C1P02/V2"  # /Armeo/002_CorJea_20210409_2_a.xdf"
    )

    visit_dir = "../dat/ReArm.lnk/DATA_named/C1P01/V1"
    # visit_dir = "../dat/ReArm.lnk/DATA_named/C1P21/V1"
    # visit_dir = "../dat/ReArm.lnk/DATA_named/C1P23/V1"
    # visit_dir = "../dat/ReArm.lnk/DATA_named/C1P20/V2"

    # visit_dir = "../dat/ReArm.lnk/DATA_named/C1P38/V1"

    os.remove(os.path.join(visit_dir, "panu.log"))
    get_panu_in_visit(visit_dir, directories_to_skip=["old", "Training", "Armeo"])